# Mini-Project 1 - Synaptic Proteins

## Overview

In this mini-project, you will analyze super-resolution STED microscopy images of synaptic proteins under two experimental conditions: **Block** and **GluGly**. Starting from raw `.tif` files, you will build an analysis pipeline that loads and normalizes images, detects protein clusters using wavelet-based segmentation, and extracts morphological features such as area, mean intensity, eccentricity, and local density. In the final section, you will apply K-Means clustering to discover protein subtypes and compare how their proportions shift between the two conditions.

By the end of this notebook you will have gone from raw microscopy images to a quantitative, data-driven comparison of protein organization across experimental groups.

---

## Before you start

### You need a Google account

This notebook runs on **Google Colab**, which provides free cloud computing and GPU access through your Google account. You will also use Google Drive to store your dataset and results across sessions. If you do not already have a Google account, please create one before the session begins.

### A note on AI tools

At some point during these exercises, you will get stuck. That is completely normal and part of the process. Before opening ChatGPT or any other AI assistant, please raise your hand and ask one of the organizers first.

The organizers know the dataset, the biological context, and the exact intent behind each exercise. An AI tool may generate something that looks correct but subtly misleads you, and you will miss the reasoning that makes the answer meaningful. We are here, we want to help, and a two-minute conversation with us will teach you far more than a generated answer ever will.

## Most Important Thing

<img src="https://media2.giphy.com/media/v1.Y2lkPTc5MGI3NjExdmpyMHFiaWRmZW0yc3huc3czbDM5ZGI3a29pd20yZXNiODczanBmdCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/6VriQO3GFRwwBVPbi4/giphy.gif" width="400">

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

# Changes directory to the main folder
if IN_COLAB:
    from google.colab import drive
    home = "/content"
    drive.mount('/content/drive')    
else:
    home = os.path.expanduser("~")
os.chdir(home)


In [ ]:
!wget -O "{home}/images.zip" "https://s3.valeria.science/flclab-foundation-models/images.zip"
!unzip -n "{home}/images.zip" -d "{home}"

# Introduction
In this mini-project we will aim to reproduce some of the results from *Wiesner et al. (Frontiers in Neurophotonics, 2020)*, at a smaller scale. Namely, we are interested in the morphological differences between synaptic proteins in neuronal cultures from a control condition, and from a condition in which chemical long-term depression (LTD) was induced. As a brief summary, the analysis pipeline used to analyze the synaptic protein data in the paper is schematically depicted in the cell below:

In [ ]:
from IPython.display import Image

image_path = f'{home}/images/MP1_pipeline.png'
display(Image(filename=image_path))

In [ ]:
image_path = f'{home}/images/manual_feature_extraction.png'
display(Image(filename=image_path))

In [ ]:
image_path = f'{home}/images/kmeans.png'
display(Image(filename=image_path))

In [ ]:
image_path = f'{home}/images/multidimensional_analysis.png'
display(Image(filename=image_path))

### We'll now implement these steps in detail, one by one. Par-Tay!

# Import necessary libraries

In [ ]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import glob
import os
import shutil
import urllib.request
import zipfile

from pathlib import Path
from typing import List, Tuple, Dict

if IN_COLAB:
    from google.colab import drive
    drive.mount(f'{home}/drive')


# Downloading synaptic protein data

In [ ]:
import os

zip_path = f"{home}/Bassoon-PSD95.zip"
zip_url = "https://s3.valeria.science/flclab-foundation-models/evaluation-data/Bassoon-PSD95.zip"

if not os.path.isfile(zip_path):
  print(f"Downloading {zip_url} to {zip_path}")
  !wget -O "$zip_path" "$zip_url"
else:
  print(f"Zip file already exists at {zip_path}")

if not os.path.isdir(f"{home}/Bassoon-PSD95"):
    print(f"Unzipping {zip_path} to {f'{home}/Bassoon-PSD95'}")
    !unzip "$zip_path" -d "{home}/"
else:
    print(f"Extracted directory already exists at '{home}/Bassoon-PSD95'")

# More imports and package installs

In [ ]:
!pip install git+https://github.com/FLClab/TiffWrapper.git
!pip install xlsxwriter

In [ ]:
from tiffwrapper import make_composite

# Preamble code

In [ ]:
import numpy
import math
import sys
import xlsxwriter
import os
from typing import List
from skimage.measure import regionprops, label
from skimage import feature, filters
from matplotlib import pyplot
from math import atan2
from skimage import morphology
from scipy.ndimage import convolve
from scipy.spatial.distance import cdist


"""
This script is intended to perform a cluster analysis based on the paper

Mapping molecular assemblies with fluorescence microscopy and object-based spatial statistics
Thibault Lagache, Alexandre Grassart, Stéphane Dallongeville, Orestis Faklaris,
Nathalie Sauvonnet, Alexandre Dufour, Lydia Danglot & Jean-Christophe Olivo-Marin

The first step is to perform the detection of the clusters with wevelet transformation
of the image and statistical thresholding of wavelets coefficients.
The second sted is to characterize the spatial distribution of the clusters using
a Marked Point Process.
The third and final step is to characterize the spatial relations between the clusters.
The Ripley's K Function will be of great help.
"""

def filter_spots(mask):
        """
        Removes the spots that are too small or too linear (using parameters min_size and min_axis)
        :param mask: 3D binary mask of spots to filter
        :return: Filtered 3D binary mask
        """
        out_mask = numpy.copy(mask).astype(bool)
        img = morphology.remove_small_objects(out_mask, min_size=30)
        mask_lab, num = label(img, connectivity=1, return_num=True)
        mask_props = regionprops(mask_lab)
        for p in mask_props:

            if p.minor_axis_length < 3:
                mask_lab[mask_lab == p.label] = 0

        out_mask = mask_lab > 0
        return out_mask

def detect_spots(img: numpy.ndarray, J_list: List[int] = (3,4), scale_threshold: float = 2.0) -> numpy.ndarray:
    """
    Detects spots in an image using the wavelet transform and statistical thresholding of wavelet coefficients.
    """
    spots_image = numpy.ndarray(img.shape)
    for ch in range(img.shape[0]):
        detector = DetectionWavelets(img[ch], J_list, scale_threshold)
        spots_image[ch] = filter_spots(detector.computeDetection())
    summed_img = np.sum(img, axis=0)
    filt = filters.gaussian(summed_img, sigma=7)
    threshold = np.mean(filt)
    filt[filt < threshold] = 0
    filt[filt >= threshold] = 1
    dendrite_mask = morphology.binary_closing(filt)

    return spots_image.astype(numpy.uint8) * dendrite_mask


class DetectionWavelets:
    """
    This is based on the paper
        "Extraction of spots in biological images using multiscale products"
    All functions from the Java code for the Icy Spot Detector plugin are implemented here.
    """

    def __init__(self, img, J_list=(3,4), scale_threshold=200):
        """Init function
        :param img: A numpy 2D array
        :param J_list: List of all chosen scales
        :param scale_threshold: Percent modifier of wavelet image threshold
        """
        self.img = img
        self.J = max(J_list)
        self.J_list = J_list
        self.scale_threshold = scale_threshold

    def computeDetection(self):
        """
        Computes the binary correlation image
        :return image_out: numpy array representing the binary image
        """

        data_in = numpy.copy(self.img).astype('float32')
        scales = self.b3WaveletScales2D(data_in)
        coefficients = self.b3WaveletCoefficients2D(scales, data_in)
        for i in range(len(coefficients)-1):
            coefficients[i] = self.filter_wat(coefficients[i], i)
        coefficients[-1] *= 0

        binary_detection_result = self.spot_construction(coefficients)
        binary_detection_result[binary_detection_result != 0] = 255

        return binary_detection_result.astype('uint8')

    def b3WaveletScales2D(self, data_in):
        """
        Computes the convolution images for scales J
        :param data_in: Base image as 1D list
        :return res_array: List of convoluted images as 1D lists
        """

        prev_array = data_in.copy()
        res_array = []

        for s in range(1, self.J+1):
            stepS = 2**(s-1)

            current_array = self.filter_and_swap(prev_array, stepS)

            if s == 1:
                prev_array = current_array
            else:
                tmp = current_array
                prev_array = tmp

            current_array = self.filter_and_swap(prev_array, stepS)
            tmp = current_array
            prev_array = tmp

            res_array.append(prev_array)

        return res_array


    def b3WaveletCoefficients2D(self, scale_coefficients, original_image):
        """
        Computes the difference between consecutive wavelet transform images
        :param scale_coefficients: List of  convoluted images as 2D numpy arrays
        :param original_image: Original image as 1D list
        :return wavelet_coefficients: List of coefficient images 2D numpy arrays
        """

        wavelet_coefficients = []
        iter_prev = original_image.copy()
        for j in range(self.J):
            iter_current = scale_coefficients[j]
            w_coefficients = iter_prev - iter_current
            wavelet_coefficients.append(w_coefficients)
            iter_prev = iter_current
        wavelet_coefficients.append(scale_coefficients[self.J-1])
        wavelet_coefficients = numpy.stack(wavelet_coefficients, 0)
        return wavelet_coefficients

    def filter_wat(self, data, depth):
        """
        Applies a threshold on the coefficient images
        :param data: image data
        :param depth: number of scale
        :param width: image width
        :param height: image height
        :return output: filtered image
        """

        output = data.copy()
        lambdac = []

        for i in range(self.J + 2):
            lambdac.append(numpy.sqrt(2 * numpy.log(data.size / (1 << (2 * i)))))

        # mad
        size = data.size
        mean = numpy.mean(data)
        s = data - mean
        a = numpy.sum(numpy.abs(s))

        mad = a / size

        dcoeff = self.scale_threshold

        coeff_thr = (lambdac[depth + 1] * mad) / dcoeff

        output[output < coeff_thr] = 0

        return output

    def spot_construction(self, input_coefficients):
        """
        Reconstructs correlation image with multiscale product
        :param input_coefficients: 3D numpy of array wavelet coefficient images
        :return output: Correlation image as 2D numpy array
        """
        J_array = numpy.array(self.J_list)-1
        #zero_coords = numpy.prod(input_coefficients[J_array], axis=0) > 0

        output = numpy.prod(input_coefficients[J_array], axis=0)
        #output *= zero_coords
        return output

    @staticmethod
    def filter_and_swap(array_in, stepS):
        kernel = numpy.array([1/16, 1/4, 3/8, 1/4, 1/16])
        inter = numpy.array([1, 2, 3, 4])
        for s in range(stepS):
            if s > 0:
                kernel = numpy.insert(kernel, inter, 0)
                inter = inter + numpy.array([1, 2, 3, 4])
        kernel_base = numpy.zeros((kernel.size, kernel.size))
        kernel_base[int((kernel.size-1)/2)] = kernel
        new_img = convolve(array_in, kernel_base)
        new_img = numpy.transpose(new_img)

        return new_img


class SpatialDistribution:
    """ This is based on the Marked Point Process as shown in the cited above paper
    Marked is the attributes of the cluster (shape, size, color)
    Point Process is the position of the clusters (centroid)

    :returns : A list of tuple (regionprops, area, centroid)
    NOTE. Centroid is (y, x) coordinates
    """
    def __init__(self, prob_map, img, cs=0, min_axis=1):
        """This is the init function
        :param prob_map: A 2D numpy array of boolean detected clusters
        :param img: numpy array, image data used for intensity weighting for centroid
        :param cs: Minimum area of a cluster
        """
        self.img = img
        self.P = prob_map
        self.P[self.P > 0] = 1
        self.cs = cs
        self.min_axis = min_axis

    def mark(self):
        """
        This function creates the Marked Point Processed of every cluster
        :return mark: List of every spot in the current channel as (regionprops, area, centroid) tuples
        """

        # Label each spot for regionprops
        labels = label(self.P, connectivity=2)

        props = regionprops(labels, intensity_image=self.img)
        mark = []
        for p in props:
            labimage = p.image
            min_distance = int(0.08 / 0.015) // 2 + 1
            peaks = feature.peak_local_max(self.img, min_distance=min_distance, exclude_border=False, labels=labimage)

            s = p.area
            if s > 0:
                try:
                    try:
                        cent_int = p.weighted_centroid
                        int(cent_int[0])
                    except ValueError:
                        cent_int = p.centroid

                    # Reject small and linear clusters
                    if s >= self.cs and p.minor_axis_length >= self.min_axis \
                            and p.major_axis_length >= self.min_axis\
                            and p.perimeter > 0:
                        mark.append(
                            (p, s, cent_int, len(peaks))  # p is the regionprops for a spot: every characteristic can be accessed
                                              # as an attribute e.g. p.eccentricity
                        )
                except IndexError:
                    print('Index error: spot was ignored.')

        return mark

    def poly_area(self, x, y):
        """ This function computes the area of a cluster
        :param x: A numpy array of x coordinates
        :param y: A numpy array of y coordinates
        """
        return 0.5*numpy.abs(numpy.dot(x,numpy.roll(y,1))-numpy.dot(y,numpy.roll(x,1)))


class SpatialRelations:
    """ This is based on the the paper cited above
    To characterise the spatial relations between two populations A1 (green) and A2 (red) of
    objects (spots or localisations), we use the Ripley’s K function, a gold standard for analysing
    the second-order properties (i.e., distance to neighbours) of point processes.
    """
    def __init__(self, MPP1, MPP2, sROI, roivolume, poly, img, n_rings, step, filename):
        """ The init function
        :param MPP1: The marked point process of every clusters in the first channel
        :param MPP2: The marked point process of every clusters in the second channel
        :param sROI: A tuple (y, x) of the window size
        :param roivolume: Volume of the detected ROI
        :param poly: List of vertices of the ROI
        :param img: numpy array containing image data
        :param n_rings: int, number of rings around each spot
        :param step: numeric, width of each ring
        :param filename: string, name of the current image, used for naming output files
        """

        self.img = img
        self.filename = filename

        self.MPP1 = MPP1
        self.MPP2 = MPP2

        self.ROIarea = roivolume
        self.poly = poly

        # self.MPP1_ROI = self.mpp_in_contours(self.MPP1)
        # self.MPP2_ROI = self.mpp_in_contours(self.MPP2)
        self.MPP1_ROI = self.MPP1
        self.MPP2_ROI = self.MPP2

        self.sROI = sROI
        self.max_dist = n_rings*step
        self.pas = step

        self.rings = numpy.array([r*step for r in range(0, n_rings+1, 1)])
        self.imgw1 = self.image_windows(self.MPP1_ROI)
        self.imgw2 = self.image_windows(self.MPP2_ROI)

        self.neighbors = self.nearest_neighbors()

        self.distance_fit, self.N_fit = self.dist_fit()

    @staticmethod
    def matrix_dist(X, Y):
        """
        Calculates the distance between every point contained in matrices
        :param X: numpy array with shape (len(MPP1), 2), coordinates of every points in channel 1
        :param Y: numpy array with shape (len(MPP2), 2), coordinates of every points in channel 2
        :return: numpy array of distances between every combination of points in the two channels
        """
        return numpy.sqrt((X[:, 0] - Y[:, 0]) ** 2 + (X[:, 1] - Y[:, 1]) ** 2)

    def nearest_neighbors(self):
        """
        Finds the nearest neighbor in both MPPs of each spot and the distance between them
        :return: List of tuples (dist. to neighbor, neighbor index) for each combination of channels
        """

        mpp1_data = numpy.zeros((len(self.MPP1_ROI), 2))
        mpp2_data = numpy.zeros((len(self.MPP2_ROI), 2))

        for i in range(len(self.MPP1_ROI)):
            p, s, (y, x), n_peaks = self.MPP1_ROI[i]
            mpp1_data[i, 0] = x
            mpp1_data[i, 1] = y

        for i in range(len(self.MPP2_ROI)):
            p, s, (y, x), n_peaks = self.MPP2_ROI[i]
            mpp2_data[i, 0] = x
            mpp2_data[i, 1] = y

        # MPP1 with MPP1
        distances = []
        min_distance = []
        for x in mpp1_data:
            repmat = numpy.tile(x, (mpp1_data.shape[0], 1))
            distance = self.matrix_dist(repmat, mpp1_data)
            distance_pos = distance[numpy.where((distance > 0))]
            distances.append(distance)
            m = distance_pos.min()
            m_index = numpy.where((distance == m))[0][0]
            p2, s2, (y2, x2), n_peaks2 = self.MPP1_ROI[m_index]
            delta_x = x2 - x[0]
            delta_y = y2 - x[1]
            angle = atan2(delta_y, delta_x)
            min_distance.append((m, m_index, angle))
        min_dist_11 = min_distance

        # MPP1 with MPP2
        distances = []
        min_distance = []
        for x in mpp1_data:
            repmat = numpy.tile(x, (mpp2_data.shape[0], 1))
            distance = self.matrix_dist(repmat, mpp2_data)
            distance_pos = distance[numpy.where((distance > 0))]
            distances.append(distance)
            m = distance_pos.min()
            m_index = numpy.where((distance == m))[0][0]
            p2, s2, (y2, x2), n_peaks2 = self.MPP2_ROI[m_index]
            delta_x = x2 - x[0]
            delta_y = y2 - x[1]
            angle = atan2(delta_y, delta_x)
            min_distance.append((m, m_index, angle))
        min_dist_12 = min_distance

        # MPP2 with MPP1
        distances = []
        min_distance = []
        for x in mpp2_data:
            repmat = numpy.tile(x, (mpp1_data.shape[0], 1))
            distance = self.matrix_dist(repmat, mpp1_data)
            distance_pos = distance[numpy.where((distance > 0))]
            distances.append(distance)
            m = distance_pos.min()
            m_index = numpy.where((distance == m))[0][0]
            p2, s2, (y2, x2), n_peaks2 = self.MPP1_ROI[m_index]
            delta_x = x2 - x[0]
            delta_y = y2 - x[1]
            angle = atan2(delta_y, delta_x)
            min_distance.append((m, m_index, angle))
        min_dist_21 = min_distance

        # MPP2 with MPP2
        min_distance = []
        distances = []
        for x in mpp2_data:
            repmat = numpy.tile(x, (mpp2_data.shape[0], 1))
            distance = self.matrix_dist(repmat, mpp2_data)
            distance_pos = distance[numpy.where((distance > 0))]
            distances.append(distance)
            m = distance_pos.min()
            m_index = numpy.where((distance == m))[0][0]
            p2, s2, (y2, x2), n_peaks2 = self.MPP2_ROI[m_index]
            delta_x = x2 - x[0]
            delta_y = y2 - x[1]
            angle = atan2(delta_y, delta_x)
            min_distance.append((m, m_index, angle))
        min_dist_22 = min_distance

        return min_dist_11, min_dist_12, min_dist_21, min_dist_22

    def dist_fit(self):
        """
        Allows to generate the distance_fit and N_fit as in the paper
        :return distance_fit: List of distances for every ring
        :return N_fit: Number of rings
        """
        distance_fit = [0]
        temp = distance_fit[0]
        while temp + self.pas <= self.max_dist:
            temp += self.pas
            distance_fit.append(temp)
        N_fit = len(distance_fit)
        if N_fit == 1:
            distance_fit.append(self.max_dist)
            N_fit = len(distance_fit)

        return distance_fit, N_fit

    def image_windows(self, points):
        """
        This function implements the image windows as proposed in the java script
        version of the algorithm.
        """
        h, w = self.sROI
        imagewindows = [[[] for i in range(int(h / self.max_dist) + 1)] for j in range((int(w / self.max_dist) + 1))]
        dist_to_border = self.nearest_contour(points)
        for k in range(len(points)):
            p, s, (y, x), n_peaks = points[k]
            j, i = int(y / self.max_dist), int(x / self.max_dist)
            imagewindows[i][j].append((y, x, s, dist_to_border[k]))
        return imagewindows

    def correlation_new(self):
        """
        This function computes the G vector of Ripley's function as in Ripley2D.java
        results[][1]=K et results[][2]=moyenne distances results[][3]=moyenne
        distances^2
        :return result: numpy array in which the first row is Ripley's G matrix
        """
        result = numpy.zeros((3, self.N_fit - 1))
        delta_K = numpy.zeros(self.N_fit - 1)
        d_mean = numpy.zeros(self.N_fit - 1)
        d_mean_2 = numpy.zeros(self.N_fit - 1)
        count = numpy.zeros(self.N_fit - 1)
        ROIy, ROIx = self.sROI
        n1, n2 = len(self.MPP1_ROI), len(self.MPP2_ROI)
        for i in range(len(self.imgw1)):
            for j in range(len(self.imgw1[i])):
                for y1, x1, s1, db1 in self.imgw1[i][j]:
                    # d = self.distance_pl(self.poly,x1,y1)
                    # d = self.dist_to_contour(x1, y1)
                    d = db1
                    # d = min(x1, ROIx - x1, y1, ROIy - y1) # min distance from ROI
                    for k in range(max(i - 1, 0), min(i + 1, len(self.imgw1) - 1) + 1):
                        for l in range(max(j - 1, 0), min(j + 1, len(self.imgw1[i]) - 1) + 1):
                            for y2, x2, s2, db2 in self.imgw2[k][l]:
                                temp = self.distance(x1,x2,y1,y2) # distance
                                if (y1, x1, s1) != (y2, x2, s2):
                                    weight = 1 # weight
                                    if temp > d:
                                        weight = 1 - (numpy.arccos(d / temp)) / numpy.pi
                                    for m in range(1, self.N_fit):
                                        if (temp < self.distance_fit[m]) & (temp >= self.distance_fit[0]):
                                            delta_K[m - 1] += (1 / weight) * self.ROIarea / (n1 * n2)
                                            count[m - 1] += 1
                                            d_mean[m - 1] += temp
                                            d_mean_2[m - 1] += temp**2
                                            break

        for l in range(self.N_fit - 1):
            result[0, l] = delta_K[l]
            if count[l] > 0:
                result[1, l] = d_mean[l] / count[l]
                result[2, l] = d_mean_2[l] / count[l]
            else:
                result[1, l] = 0
                result[2, l] = 0
        return result

    def nearest_contour(self, mpp):
        """
        Finds the distance between every point and the ROI border
        Because the distance is point to point, the ROI must have points all along its contour for this to be accurate.
        :param mpp: Marked Point Process of a single channel (list of points)
        :return: List of distances
        """
        mpp_data = numpy.zeros((len(mpp), 2))

        poly_sum = numpy.concatenate(self.poly)

        poly_data = numpy.zeros((len(poly_sum), 2))

        for i in range(len(mpp)):
            p, s, (y, x), n_peaks = mpp[i]
            mpp_data[i, 0] = x
            mpp_data[i, 1] = y

        for i in range(len(poly_sum)):
            y, x = poly_sum[i]
            poly_data[i, 0] = x
            poly_data[i, 1] = y

        distances = []
        min_distance = []
        for x in mpp_data:
            repmat = numpy.tile(x, (poly_data.shape[0], 1))
            distance = self.matrix_dist(repmat, poly_data)
            distances.append(distance)
            m = distance.min()
            min_distance.append(m)

        return min_distance

    def dist_to_contour(self, x, y):
        '''
        (Unused alternative to nearest_contour)
        Meant to calculate the distance between a point and the edge of a ROI if
        the ROI is composed of multiple contours
        :param x: x coordinate of the point
        :param y: y coordinate of the point
        :return: Minimum distance between a point and the contour of the ROI
        '''
        dist = sys.maxsize
        min_dist = dist

        for c in self.poly:
            min_dist = self.distance_pl(c, x, y)
            min_dist = min(dist, min_dist)

        return min_dist

    def mpp_in_poly(self, MPP, poly):
        """
        This function checks which spots are in the selected polygon
        :param MPP: Marked Point Process; list of marks
        :param poly: A list of (x, y) coordinates corresponding to a polygon's (ROI) vertexes
        :return: List of marks in the ROI
        """
        MPP_ROI = []
        for point in MPP:
            y, x = point[2]

            n = len(poly)
            inside = False

            p1y, p1x = poly[0]
            for i in range(n + 1):
                p2y, p2x = poly[i % n]
                if y > min(p1y, p2y):
                    if y <= max(p1y, p2y):
                        if x <= max(p1x, p2x):
                            if p1y != p2y:
                                xints = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                            else:
                                xints = None
                            if p1x == p2x or x <= xints:
                                inside = not inside
                p1x, p1y = p2x, p2y
            if inside:
                MPP_ROI.append(point)

        return MPP_ROI

    def mpp_in_contours(self, MPP):
        """
        Calls mpp_in_poly in a loop for every region of the ROI if it's composed of multiple shapes
        :param MPP: Marked Point Process; list of marks
        :return: List of spots inside ROI
        """

        spots_in_contour = []
        for poly in self.poly:
            spots_in_contour += self.mpp_in_poly(MPP, poly)

        return spots_in_contour

    def variance_theo_delta_new(self):
        """
        This function implements the computation of the standard deviation of G
        :return result: numpy array, variance matrix
        """
        n1, n2 = len(self.MPP1_ROI), len(self.MPP2_ROI)
        mu = self.mean_G()
        result = numpy.zeros(self.N_fit - 1)
        results, N_h = self.beta_correction(self.max_dist/10, 100)

        for k in range(1, self.N_fit):
            distancek_1 = self.distance_fit[k - 1]
            distancek = self.distance_fit[k]

            d2 = distancek_1**2
            d2bis = distancek**2
            e1 = numpy.pi * (distancek_1**2)
            e2 = numpy.pi * (distancek**2)

            temp_A1, temp_A2, temp_A3 = 0, 0, 0

            sum_h_a, sum_h_a_bis = 0, 0

            for i in range(len(self.imgw1)):
                for j in range(len(self.imgw1[i])):
                    for y1, x1, s1, db1 in self.imgw1[i][j]:
                        dist = db1
                        if (dist < distancek) & (dist >= self.distance_fit[0]):
                            sum_h_a += results[math.ceil(N_h * dist / distancek)]
                        else:
                            sum_h_a += 1
                        if k > 1:
                            if (dist < distancek_1) & (dist > self.distance_fit[0]):
                                sum_h_a_bis += results[math.ceil(N_h * dist / distancek_1)]
                            else:
                                sum_h_a_bis += 1

                        for m in range(max(i - 1, 0), min(i + 1, len(self.imgw1) - 1) + 1):
                            for l in range(max(j - 1, 0), min(j + 1, len(self.imgw1[i]) - 1) + 1):
                                for y2, x2, s2, db2 in self.imgw1[m][l]:
                                    distance_ij = self.distance(x1, x2, y1, y2)
                                    if distance_ij > 0:
                                        if distance_ij < 2 * distancek_1:
                                            temp1 = 2 * d2 * numpy.arccos(distance_ij / (2 * distancek_1))
                                            temp2 = 0.5 * distance_ij * numpy.sqrt(4 * d2 - distance_ij**2)
                                            temp_A1 += temp1 - temp2
                                        if distance_ij < 2 * distancek:
                                            temp1 = 2 * distancek**2 * numpy.arccos(distance_ij / (2 * distancek))
                                            temp2 = 0.5 * distance_ij * numpy.sqrt(4 * distancek**2 - distance_ij**2)
                                            temp_A2 += temp1 - temp2
                                        if distance_ij < distancek_1 + distancek:
                                            if distance_ij + distancek_1 < distancek:
                                                temp_A3 += 2 * numpy.pi * d2
                                            else:
                                                temp1 = d2 * numpy.arccos((distance_ij**2 + d2 - d2bis) / (2 * distance_ij * distancek_1))
                                                temp2 = d2bis * numpy.arccos((distance_ij**2 + d2bis - d2) / (2 * distance_ij * distancek))
                                                temp3 = 0.5 * numpy.sqrt((- distance_ij + distancek_1 + distancek)
                                                                         * (distance_ij - distancek_1 + distancek)
                                                                         * (distance_ij + distancek_1 - distancek)
                                                                         * (distance_ij + distancek_1 + distancek))
                                                temp_A3 += 2 * (temp1 + temp2 - temp3)
            I2 = (temp_A1 + temp_A2 - temp_A3 - (e1**2 / self.ROIarea + e2**2 / self.ROIarea - 2 * e1 * e2 / self.ROIarea) * (n1 * (n1 - 1))) * n2 / self.ROIarea
            I1 = (e2 * sum_h_a - e1 * sum_h_a_bis - n1 * (e2 - e1)**2 / self.ROIarea) * n2 / self.ROIarea
            result[k - 1] = (self.ROIarea / (n2 * n1))**2 * (I1 + I2)
        return result

    def intersection2D_new(self):
        """
        This function computes the A matrix
        :return A: numpy array, A matrix
        """
        n1, n2 = len(self.MPP1_ROI), len(self.MPP2_ROI)
        A = numpy.zeros((self.N_fit - 1, self.N_fit - 1))
        for r in range(self.N_fit - 1):
            A[r, r] = n1
        S = numpy.zeros((self.N_fit, self.N_fit))

        for k1 in range(0, self.N_fit):
            for k2 in range(0, self.N_fit):
                m = min(self.distance_fit[k1], self.distance_fit[k2])  # create min
                M = max(self.distance_fit[k1], self.distance_fit[k2])  # create max

                for i in range(len(self.imgw1)):
                    for j in range(len(self.imgw1[i])):
                        for y1, x1, s1, db1 in self.imgw1[i][j]:
                            for o in range(max(i - 1, 0), min(i + 1, len(self.imgw1) - 1) + 1):
                                for l in range(max(j - 1, 0), min(j + 1, len(self.imgw1[i]) - 1) + 1):
                                    for y2, x2, s2, db2 in self.imgw1[o][l]:
                                        d = self.distance(x1, x2, y1, y2)
                                        if d > 0:
                                            if m + d < M:
                                                S[k1, k2] = S[k1, k2] + numpy.pi * m ** 2
                                            else:
                                                if d < (m + M):
                                                    temp1 = m ** 2 * numpy.arccos((d ** 2 + m ** 2 - M ** 2) / (2 * d * m))
                                                    temp2 = M ** 2 * numpy.arccos((d ** 2 + M ** 2 - m ** 2) / (2 * d * M))
                                                    temp3 = 0.5 * numpy.sqrt((-d + m + M) * (d + m - M) * (d - m + M) * (d + m + M))
                                                    S[k1, k2] = S[k1, k2] + temp1 + temp2 - temp3

        for i in range(self.N_fit - 1):
            for j in range(self.N_fit - 1):
                vol = numpy.pi * (self.distance_fit[j + 1] ** 2 - self.distance_fit[j] ** 2)
                A[i, j] = A[i, j] + (S[i + 1, j + 1] + S[i, j] - S[i, j + 1] - S[i + 1, j]) / vol
        return A

    def reduced_Ripley_vector(self, **kwargs):
        """
        Vector G0 from paper. This function returns the estimation of the coupling probability
        """
        n1, n2 = len(self.MPP1_ROI), len(self.MPP2_ROI)
        if kwargs:
            G = kwargs["G"][0, :]
            var = kwargs["var"]
            A = kwargs["A"]
        else:
            G = self.correlation_new()[0, :]
            var = self.variance_theo_delta_new()
            A = self.intersection2D_new()
        mean = self.mean_G()

        A_b = A/n1

        G0 = numpy.dot(numpy.linalg.inv(A_b), G - mean) / numpy.sqrt(var)

        return G0

    def draw_G0(self, G0):
        """
        Displays a bar graph of G0 by ring distance with a line for the G0 threshold, like figure 1d of the paper
        :param G0: numpy array, G0 matrix
        """
        T = numpy.sqrt(2 * numpy.log(len(G0)))
        pyplot.bar(self.rings[1:], G0)
        pyplot.axhline(y=T, color='red', linestyle='dashed')
        pyplot.title("G0")
        pyplot.show()

    def main2D_corr(self, G, var, A):
        """
        Currently unused
        P calculation from the SODA code (non_parametric_object.java)
        Works; very nearly gives the same results as coupling_prob
        """
        n1, n2 = len(self.MPP1_ROI), len(self.MPP2_ROI)
        delta_K = G[0,:]
        sigma_T = numpy.zeros([self.N_fit - 1])
        Num = delta_K * (n1*n2/self.ROIarea)
        mu_tab = self.mean_G() * (n1*n2/self.ROIarea)
        for p in range(self.N_fit-1):
            if p > 0:
                T_p = (numpy.sqrt(2 * numpy.log(p+1)))
            else:
                T_p = numpy.sqrt(2)
            sigma_T[p] = (n1*n2*T_p/self.ROIarea)*numpy.sqrt(var[p])

        A_matb = A * 1/n1
        A_matb_inverse = numpy.linalg.inv(A_matb)
        C = numpy.dot(A_matb_inverse, (Num - mu_tab))

        for p in range(self.N_fit-1):
            if C[p]<sigma_T[p]:
                C[p] = 0

        proba_dist = []
        for i in range(self.N_fit-1):
            if Num[i] > 0:
                proba_dist.append(C[i]/Num[i])
            else:
                proba_dist.append(0)

        return proba_dist

    def coupling_prob(self, **kwargs):
        """
        This function computes the coupling probability between the two channels
        """
        n1, n2 = len(self.MPP1_ROI), len(self.MPP2_ROI)
        if kwargs:
            G = kwargs["G"][0, :]
            var = kwargs["var"]
            A = kwargs["A"]
            G0 = kwargs["G0"]
        else:
            G = self.correlation_new()[0, :]
            var = self.variance_theo_delta_new
            A = self.intersection2D_new()
            G0 = self.reduced_Ripley_vector(**kwargs)
        mean = self.mean_G()

        # T = [numpy.sqrt(2 * numpy.log(i+1)) if i > 0 else numpy.sqrt(2) for i in range(self.N_fit)]
        T = numpy.sqrt(2 * numpy.log(len(G0)))
        coupling = numpy.array([
            (numpy.sqrt(var[i]) * G0[i]) / G[i] if G0[i] > T else 0 for i in range(len(self.rings)-1)
        ])

        # Create distance matrix and coupling probability matrix using cdist
        MPP1_array = numpy.ndarray((n1, 2))
        for i, (c1, s1, (y1, x1), n_peaks1) in enumerate(self.MPP1_ROI):
            MPP1_array[i] = numpy.array([y1, x1])

        MPP2_array = numpy.ndarray((n2, 2))
        for i, (c1, s1, (y2, x2), n_peaks2) in enumerate(self.MPP2_ROI):
            MPP2_array[i] = numpy.array([y2, x2])

        dist_array = cdist(MPP1_array, MPP2_array)
        prob_array = numpy.zeros(dist_array.shape)
        for i in range(len(self.rings) - 1):
            prob_array[numpy.where(numpy.logical_and(self.rings[i] <= dist_array, dist_array < self.rings[i+1]))] = coupling[i]
        prob_array[dist_array == 0] = 0

        # Create the list of lines for the couples excel file
        n_couples = numpy.sum(prob_array > 0)
        prob_write = []
        for i in range(n1):
            for j in range(n2):
                if prob_array[i,j] > 0:
                    prob_write.append([MPP1_array[i][1],
                                       MPP1_array[i][0],
                                       MPP2_array[j][1],
                                       MPP2_array[j][0],
                                       dist_array[i,j],
                                       prob_array[i,j],
                                       i,
                                       j])

        if numpy.sum(prob_array) > 0:
            coupling_index = (numpy.sum(prob_array)/n1, numpy.sum(prob_array)/n2)
            mean_coupling_distance = numpy.sum(dist_array*prob_array)/numpy.sum(prob_array)
        else:
            coupling_index = (0,0)
            mean_coupling_distance = None

        return prob_write, {'n_spots_0': len(self.MPP1_ROI),
                            'n_spots_1': len(self.MPP2_ROI),
                            'coupling_index': coupling_index,
                            'mean_coupling_distance': mean_coupling_distance,
                            'coupling_probabilities': coupling,
                            'n_couples': n_couples}

    def data_boxplot(self, prob_write):
        """
        Plots a box plot of a spot property by channels and coupling
        :param prob_write: List of couples and their properties
        """
        fig, axs = pyplot.subplots(1,2)

        dataC = []
        dataU = []
        for p1, s1, (y1, x1), n_peaks in self.MPP1_ROI:
            coupled = False
            for xa, ya, xb, yb, dist, p, _, _ in prob_write:
                if (y1, x1) == (ya, xa) and p1.minor_axis_length > 0:
                    dataC.append(p1.eccentricity)
                    coupled = True
            if not coupled and p1.minor_axis_length > 0:
                dataU.append(p1.eccentricity)

        data1 = [dataC, dataU]

        dataC = []
        dataU = []
        for p1, s1, (y1, x1), n_peaks in self.MPP2_ROI:
            coupled = False
            for xa, ya, xb, yb, dist, p, _, _ in prob_write:
                if (y1, x1) == (yb, xb) and p1.minor_axis_length > 0:
                    dataC.append(p1.eccentricity)
                    coupled = True
            if not coupled and p1.minor_axis_length > 0:
                dataU.append(p1.eccentricity)

        data2 = [dataC, dataU]

        axs[0].boxplot(data1, showmeans=True, labels=['Coupled', 'Uncoupled'])
        axs[1].boxplot(data2, showmeans=True, labels=['Coupled', 'Uncoupled'])

        pyplot.title('Eccentricity')
        axs[0].set_title('Channel 0')
        axs[1].set_title('Channel 1')

        pyplot.savefig('boxplot_{}'.format(self.filename))
        pyplot.close()

    def write_spots_and_probs(self, prob_write, directory, title, channels):
        """
        Writes informations about couples and single spots
        :param prob_write: list containing lists of information to write about each couple
        :param directory: string containing the path of the output file
        :param title: name of the output excel file as string
        """
        workbook = xlsxwriter.Workbook(os.path.join(directory, title), {'nan_inf_to_errors': True})
        couples = workbook.add_worksheet(name='Couples')
        titles = ['X1', 'Y1', 'X2', 'Y2', 'Distance', 'Coupling probability']

        titles = ['X1', 'Y1', 'Area_1', 'Distance to Neighbor Same Ch_1', 'Distance to Neighbor Other Ch_1', 'Eccentricity_1', 'Max intensity_1', 'Min intensity_1', 'Mean intensity_1', 'Major axis length_1', 'Minor axis length_1', 'Orientation_1', 'Perimeter_1',
                  'X2', 'Y2', 'Area_2', 'Distance to Neighbor Same Ch_2', 'Distance to Neighbor Other Ch_2', 'Eccentricity_2', 'Max intensity_2', 'Min intensity_2', 'Mean intensity_2', 'Major axis length_2', 'Minor axis length_2', 'Orientation_2', 'Perimeter', 'Coupling Distance', 'Coupling probability']


        for t in range(len(titles)):
            couples.write(0, t, titles[t])

        row = 1
        for p_list in prob_write:
            idx_1, idx_2 = p_list[6], p_list[7]
            (p1, s1, (y1, x1), n_peaks1) = self.MPP1_ROI[idx_1]
            (p2, s2, (y2, x2), n_peaks2) = self.MPP2_ROI[idx_2]

            dnn1_1, nn1, angle = self.neighbors[0][idx_1]
            dnn2_1, nn2, angle = self.neighbors[1][idx_1]

            dnn1_2, nn1, angle = self.neighbors[3][idx_2]
            dnn2_2, nn2, angle = self.neighbors[2][idx_2]

            for index in range(len(p_list)):
                datarow =[x1, y1, s1, dnn1_1, dnn2_1, p1.eccentricity,
                         p1.max_intensity, p1.min_intensity, p1.mean_intensity,
                         p1.major_axis_length, p1.minor_axis_length, p1.orientation,
                         p1.perimeter,

                         x2, y2, s2, dnn1_2, dnn2_2, p2.eccentricity,
                         p2.max_intensity, p2.min_intensity, p2.mean_intensity,
                         p2.major_axis_length, p2.minor_axis_length, p2.orientation,
                         p2.perimeter,

                         p_list[4], p_list[5]
                         ]
                for i in range(len(datarow)):
                    couples.write(row, i, datarow[i])
            row += 1

        spots1 = workbook.add_worksheet(name=f"Spots ch{channels[0]}")
        spots1_array = []
        spots2 = workbook.add_worksheet(name=f"Spots ch{channels[1]}")
        spots2_array = []
        titles = ['X1', 'Y1', 'Area', 'Distance to Neighbor Same Ch', 'Distance to Neighbor Other Ch', 'Eccentricity',
                  'Max intensity', 'Min intensity', 'Mean intensity',
                  'Major axis length', 'Minor axis length', 'Orientation',
                  'Perimeter', 'Peaks', 'Coupled', 'Coupling probability']

        for t in range(len(titles)):
            spots1.write(0, t, titles[t])
        row = 1
        for (p1, s1, (y1, x1), n_peaks) in self.MPP1_ROI:
            coupled = 0
            coupling_prob = 0
            for xa, ya, xb, yb, dist, p, _, _ in prob_write:
                if (y1, x1) == (ya, xa):
                    coupled = 1
                    coupling_prob = p
            dnn1, nn1, angle = self.neighbors[0][row-1]
            dnn2, nn2, angle = self.neighbors[1][row-1]

            datarow = [x1, y1, s1, dnn1, dnn2, p1.eccentricity,
                       p1.max_intensity, p1.min_intensity, p1.mean_intensity,
                       p1.major_axis_length, p1.minor_axis_length, p1.orientation,
                       p1.perimeter, n_peaks, coupled, coupling_prob]

            spots1_array.append([0, *datarow])
            for i in range(len(datarow)):
                spots1.write(row, i, datarow[i])

            row += 1

        titles = ['X2', 'Y2', 'Area', 'Distance to Neighbor Same Ch', 'Distance to Neighbor Other Ch', 'Eccentricity',
                  'Max intensity', 'Min intensity', 'Mean intensity',
                  'Major axis length', 'Minor axis length', 'Orientation',
                  'Perimeter', 'Peaks', 'Coupled', 'Coupling probability']
        titles_array = ["Channel", "X", "Y", "Area", "Distance to Neighbor Same Ch", "Distance to Neighbor Other Ch", "Eccentricity",
                  "Max intensity", "Min intensity", "Mean intensity",
                  "Major axis length", "Minor axis length", "Orientation",
                  "Perimeter", "Peaks", "Coupled", "Coupling probability"]
        for t in range(len(titles)):
            spots2.write(0, t, titles[t])
        row = 1

        for p2, s2, (y2, x2), n_peaks in self.MPP2_ROI:
            coupled = 0
            coupling_prob = 0
            for xc, yc, xd, yd, dist, p, _, _ in prob_write:
                if (y2, x2) == (yd, xd):
                    coupled = 1
                    coupling_prob = p
            dnn1, nn1, angle = self.neighbors[3][row-1]
            dnn2, nn2, angle = self.neighbors[2][row-1]
            datarow = [x2, y2, s2, dnn1, dnn2, p2.eccentricity,
                       p2.max_intensity, p2.min_intensity, p2.mean_intensity,
                       p2.major_axis_length, p2.minor_axis_length, p2.orientation,
                       p2.perimeter, n_peaks, coupled, coupling_prob]

            spots2_array.append([1, *datarow])
            for i in range(len(datarow)):
                spots2.write(row, i, datarow[i])
            row += 1
        try:
            workbook.close()
        except PermissionError:
            print("Warning: Workbook is open and couldn't be overwritten!")
        return spots1_array, spots2_array, titles_array

    @staticmethod
    def beta_correction(step, nbN):
        """ This function computes the boundary condition
        beta_correction(maxdist / 10, 100);
        """
        valN = 1/(1/nbN)
        N_h = int(valN) + 1

        alpha = numpy.zeros(N_h + 1)
        results = numpy.zeros(N_h + 1)
        for i in range(1, results.size):
            alpha[i] = (i / N_h)
        for i in range(results.size):
            j = 2
            h = alpha[i] + step
            while h <= 1:
                results[i] = results[i] + h * step / (1 - 1 / numpy.pi * numpy.arccos(alpha[i] / h))
                h = alpha[i] + j * step
                j += 1
            results[i] = results[i] * 2 + alpha[i] * alpha[i]
        return results, N_h

    def distance(self, x1, x2, y1, y2):
        """ This function computes the distance between two points
        :param x1: x coordinates of first point
        :param x2: x coordinates of the second point
        :param y1: y coordinates of the first point
        :param y2: y coordinates of the second point
        """
        return numpy.sqrt((x1 - x2)**2 + (y1 - y2)**2)

    def boundary_condition(self, h, x1, x2, y1, y2):
        """ This is to compute the boundary condition (eq 13) in supplementary info
        :param h: The nearest boundary
        :param x1: The x coordinate of point 1
        :param y1: The y coordinate of point 1
        :param X: A numpy 1D array of x random coordinates
        :param Y: A numpy 1D array of y random coordinates

        :returns : The boundary condition
        """
        d = self.distance(x1, x2, y1, y2)
        #minimum = numpy.array([min(h, d) for d in dist])
        if d == 0:
            d = 0.0000001
        minimum = min(h, d)

        k = (1 - 1/numpy.pi * numpy.arccos(minimum / d))
        return 1 / k

    def mean_G(self):
        """ This function computes the mean of G in 2D
        """
        mean = numpy.pi * numpy.diff(numpy.array(self.distance_fit) ** 2)
        return mean

# Exercise 1: Loading the data

In this exercise we will load the dataset of synaptic protein images. The dataset contains images from two experimental conditions (**Block**, our control neurons, and **GluGly**, our chemically-induced LTD neurons), organized as TIFF files in a hierarchical directory structure.

## 1.1 - Load the files
Implement the `list_images` function, which should simply return a list containing the path to every image in the root directory (`Bassoon-PSD95`). The print statements in the provided if-else branch tell you whether your function correctly found all 17 files. Can you figure out what proteins were imaged in the two channels?
- **Hint #1**: `os.listdir` or `glob.glob` are great functions for this. Google them and their documentation if you are not familiar with them.

In [ ]:
###### Exercise 1.1 ######
def list_images(root: str) -> List[str]:
    """Recursively returns all `.tif` files under *root* as a list of strings.

    Args:
        root - str: Root directory to search.

    Returns:
        list of strings: paths to all tiff files in the root directory.

    Example:
        >>> files = list_images(root="/content/dataset")
    """
    files = ... # TODO
    return list(set(files))


# Uncomment the lines below to test implementation
root = f'{home}/Bassoon-PSD95'
files = list_images(root=root)

if len(files) != 17:
  print(f"Try again, the number of files found is not 17 ({len(files)})")
else:
  print("All 17 files found.")


## 1.2 - Investigate one of the files.
Load the data from one of the files (chosen randomly) into a numpy array. Print the shape of the image, the minimum and maximum pixel values, the data type, and validate that the data is indeed of type numpy array.
- **Hint #1**: `tifffile.imread` to load the data into a numpy array.
- **Hint #2**: Built-in `type` function to check the data type of a variable.

In [ ]:
###### Exercise 1.2 ######
f = np.random.choice(files)
data = ... # TODO

# Uncomment the lines below to test implementation
print(...) # TODO

## 1.3 — Extract Condition Labels from File Paths

**Context:** File paths encode the experimental condition as the parent folder name (e.g. `.../Block/image.tif`). We need this folder name to separate files into their corresponding conditions for the ensuing analysis. You will implement a parser and a loader that returns both file paths and their associated conditions. The if-else branch statement at the end of the cell checks if you have the correct number of files for each condition.

**Steps:**
1. Implement `get_condition_from_filename`: extract the condition name from the full path (hint: the condition is the last folder before the filename).
2. Implement `list_image_paths_with_conditions`: use `glob.glob` to list all `.tif` files recursively, then map `get_condition_from_filename` over them.
3. How many files are there per condition?

**Key functions:** `glob.glob`, `str.split`  
**Available variables after this cell:** `files`, `conditions`

In [ ]:
###### Exercise 1.3 ######
def get_condition_from_filename(filename: str) -> str:
  """
  Get the name of the experimental condition from the filename.

  Params:
  -------
    filename - str: Full file path

  Returns:
  --------
    str: Name of the experimental condition
  """
  # TODO
  pass


def list_image_paths_with_conditions(root: str) -> Tuple[List[str], List[str]]:
  """
  Lists image paths and their associated experimental conditions.

  Params:
  -------
    root - str: Root directory

  Returns:
  --------
    Tuple[List[str], List[str]]: A tuple containing a list of file paths and a list of corresponding conditions.
  """
  files = list_images(root=f"{home}/Bassoon-PSD95")
  conditions = ... # TODO - Hint: Use the get_condition_from_filename function with list comprehension
  return files, conditions


# Uncomment the lines below to test implementation
files, conditions = list_image_paths_with_conditions(root=root)

uniques, counts = np.unique(conditions, return_counts=True)
condition_counts = {key: value for key, value in zip(uniques, counts)}

if condition_counts["Block"] != 10 or condition_counts["GluGly"] != 7:
  print(f"One of the conditions does not have the correct number of files.\n\tBlock: {condition_counts['Block']}/10\n\tGluGly: {condition_counts['GluGly']}/7")
else:
  print(f"Correctly found 10 Block files and 7 GluGly files")

# Exercise 2: Visualize Data

Now that we are familiar with the files, we must inspect the raw images before extracting features from them. We found in the previous exercise that images have 2 channels, are 8 bits, are quite large, and have a wide range of pixel values.

## 2.1 — Normalize Channels & Display Random Image

**Context:** TIFF images are stored as integer arrays with arbitrary intensity ranges. To facilitate analysis and comparison between images and conditions, we will normalize each image to the [0, 1] using robust percentiles before visualization or analysis.

**Steps:**
1. Implement `channel_normalize`: for each channel compute the qmin and qmax quantile percentile intensities (`np.quantile`), then clip and rescale to [0, 1]. `qmin=0.01` and `qmax=0.99` are usually good starting points for the quantiles, but you may see oddities in the resulting images. Try tuning the quantiles values until you are satisfied with the output.
2. Implement `display_random_image`: pick a random file, load it with `tifffile.imread`, normalize it, convert to RGB with `make_composite(img, luts['magenta', 'cyan'], ranges=[(0,1), (0,1])`, and set the experimental condition as the plot title.

Run the cell multiple times to see different images from the two experimental conditions.

**Key functions:** `np.quantile`, `np.clip`, `tifffile.imread`, `make_composite`  
**Available variables:** `files`, `get_condition_from_filename`

In [ ]:
### Exercise 2.1 ###

def channel_normalize(img: np.ndarray) -> np.ndarray:
  """
  Normalizes each channel of an input image to the range [0, 1].

  Params:
  -------
    img - np.ndarray: Input image with shape (channels, height, width).

  Returns:
  --------
    np.ndarray: Normalized image with pixel values in [0, 1].
  """
  norm_img = np.zeros_like(img).astype(np.float32)
  for ch in range(img.shape[0]):
    ### TODO
    ## 1. index the image at channel `ch`
    ## 2. Compute the lower and upper quantiles desired
    ## 3. Normalize the image using the quantiles
    ch_img = ... # TODO
    qmin = ... # TODO
    qmax = ... # TODO
    norm_img[ch] = ... # TODO
    norm_img[ch] = np.clip(norm_img[ch], 0, 1) # When using quantiles, we clip to 0 and 1, as the true min and max of the image will be outside that range
  return norm_img

def display_random_image(files: List[str]) -> None:
  """
  Loads a random image, normalizes it, converts it to RGB, and displays it with its condition as the title.

  Params:
  -------
    files - List[str]: List of image file paths.

  Returns:
  --------
    None
  """
  f = ... # TODO: Pick random file
  print(f"[---] {get_condition_from_filename(f)} image [---]")
  img = ... # TODO: Load image from file
  img = ... # TODO: Normalize image
  img = make_composite(img, luts=['magenta', 'cyan'], ranges=[(0,1), (0,1)])
  fig = plt.figure(figsize=(8,8))
  ax = fig.add_subplot(111)
  ax.set_title(...) # TODO: Set condition as title of the plot
  ax.imshow(...) # TODO: ... Display image on the ax
  ax.axis("off")
  plt.show()
  plt.close(fig)


display_random_image(files=files)

## 2.2 — Random Crops & Condition Visualization

**Context:** Synaptic protein clusters are small structures. Examining random 224×224 crops from full-field images reveals the heterogeneity of cluster morphology within and across conditions.

**Steps:**
1. Implement `take_random_crop`: sample a valid top-left corner `(random_y, random_x)` within `[0, H-crop_size] × [0, W-crop_size]` and slice the image using the coordinates sampled and the crop size to return the crop.
2. Implement `display_random_crops`: filter files by condition, load and normalize one image, generate `num_crops` crops, and display them in a 5×5 grid.
3. Compare crops between **Block** and **GluGly**. Do you see any obvious morphological differences?

**Key functions:** `np.random.randint`, `channel_normalize`, `make_composite`, `plt.subplots`  
**Available variables:** `files`, `channel_normalize`, `get_condition_from_filename`

In [ ]:
### Exercise 2.2 ###
def take_random_crop(img: np.ndarray, crop_size: int = 224) -> np.ndarray:
  """
  Extracts a random square crop from an image.

  Params:
  -------
    img - np.ndarray: Input image with shape (channels, height, width).
    crop_size - int: The size of the square crop (e.g., 224 for 224x224).

  Returns:
  --------
    np.ndarray: A random crop of the image.
  """
  H, W = img.shape[1:]
  random_y = ... # TODO
  random_x = ... # TODO
  # Our image has shape (Channels, Height, Width)
  crop = img[...] # TODO
  return crop

def display_random_crops(files: List[str], num_crops: int = 25, condition: str = "Block") -> None:
  """
  Displays a grid of random crops from images belonging to a specified condition.

  Params:
  -------
    files - List[str]: List of image file paths.
    num_crops - int: The number of random crops to display.
    condition - str: The experimental condition to filter images by.

  Returns:
  --------
    None
  """
  file_condition = None
  while file_condition != condition:
    f = np.random.choice(files)
    file_condition = get_condition_from_filename(f)
  print(f"[---] {condition} crops [---]")
  # TODO
  # 1. Load image
  img = ...
  # 2. Normalize image
  img = ...
  ## 3. Take `num_crops` crops (hint: use list comprehension)
  crops = [...]
  ## 4. Display all crops in a single figure (code started below)
  fig, axs = plt.subplots(5, 5, figsize=(7, 7))
  for idx, crop in enumerate(crops):
    ax = axs[idx // 5, idx % 5]
    crop_rgb = make_composite(...) # TODO
    ax.... # TODO
    ax.axis("off")
  plt.show()
  plt.close(fig)

### Uncomment lines below to test your code
display_random_crops(files=files, condition="Block")
display_random_crops(files=files, condition="GluGly")


# 3) Extracting Features from the Data

Our hypothesis is that synaptic protein clusters differ between **Block** and **GluGly** conditions. To test this, we quantify structural properties (area, intensity, shape) of individual clusters detected in each image.

## 3.1 — Thresholding: Isolating Protein Clusters

**Context:** To detect protein clusters we must separate bilogically relevant signal from background. Three thresholding strategies are compared: setting a manual threshold (simple), Otsu's method (data-driven), and wavelet-based spot detection (most accurate for fluorescence). The wavelet-based function is already provided, but you should implement the simple method and Otsu's method to be able to compare all three.

**Steps:**
1. Implement the `simple_threshold` function, which should simply set all pixels larger than a given threshold to 1, and all others to 0. Given a [0, 1] normalizing, you could start with a threshold of 0.5, and then try to refine the segmentation using the statistics of the image(s) (mean, standard dev, etc...)
2. Implement the `otsu_thresholding` method, which should using the `threshold_otsu` function from the `skimage.filters` package. If you are not familiar with the function, go check its documentation to use it properly.
3. Use the `threshold_comparisons` function to visually compare the three methods.


- Which method produces the cleanest segmentation?
- Which generates too many false positives?

`wavelet_thresholding` (via `detect_spots`) will be used in all subsequent exercises.

In [ ]:
from skimage.filters import threshold_otsu


### Exercise 3.1 ###
def simple_threshold(img: np.ndarray, display: bool = False) -> None:
  """
  Applies a simple threshold based on mean + 2*std to an image and displays the original and masked image.

  Params:
  -------
    img - np.ndarray: Input image.
    display - bool: Whether or not to display the image and mask

  Returns:
  --------
    np.ndarray: protein mask
  """
  mask = img > some_threshold  # TODO: replace some_threshold with your desired threshold here
  if display:
    fig, axs = plt.subplots(1, 2, figsize=(12,12))
    img_rgb = make_composite(img, luts=["magenta", "cyan"], ranges=[(0, 1), (0,1)])
    mask_rgb = make_composite(mask, luts=["magenta", "cyan"], ranges=[(0, 1), (0,1)])
    axs[0].imshow(img_rgb)
    axs[1].imshow(mask_rgb)
    for ax in axs:
      ax.axis("off")
    axs[1].set_title("Simple thresholding")
    plt.show()
    plt.close(fig)
  return mask

def otsu_thresholding(img: np.ndarray, display: bool =  False) -> np.ndarray:
  """
  Applies Otsu's thresholding method to an image and displays the original and masked image.

  Params:
  -------
    img - np.ndarray: Input image.
    display - bool: Whether or not to display the image and mask

  Returns:
  --------
    np.ndarray: protein mask
  """
  thresh_value = # TODO Use otsu to compute the threshold
  mask = img > thresh_value
  if display:
    fig, axs = plt.subplots(1, 2, figsize=(10,10))
    img_rgb = make_composite(img, luts=["magenta", "cyan"], ranges=[(0, 1), (0,1)])
    mask_rgb = make_composite(mask, luts=["magenta", "cyan"], ranges=[(0, 1), (0,1)])
    axs[0].imshow(img_rgb)
    axs[1].imshow(mask_rgb)
    for ax in axs:
      ax.axis("off")
    axs[1].set_title("Otsu thresholding")
    plt.show()
    plt.close(fig)
  return mask

def wavelet_thresholding(img: np.ndarray, display: bool = False) -> np.ndarray:
  """
  Applies wavelet-based spot detection to an image and displays the original and detected spots.

  Params:
  -------
    img - np.ndarray: Input image.
    display - bool: Whether or not to display the image and mask

  Returns:
  --------
    np.ndarray: protein mask
  """
  mask = detect_spots(img=img)
  if display:
    fig, axs = plt.subplots(1, 2, figsize=(15,15))
    img_rgb = make_composite(img, luts=["magenta", "cyan"], ranges=[(0, 1), (0,1)])
    mask_rgb = make_composite(mask, luts=["magenta", "cyan"], ranges=[(0, 1), (0,1)])
    axs[0].imshow(img_rgb)
    axs[1].imshow(mask_rgb)
    for ax in axs:
      ax.axis("off")
    axs[1].set_title("Wavelet thresholding")
    plt.show()
    plt.close(fig)
  return mask


def threshold_comparisons(files: List[str]) -> None:
  """
  Compares different thresholding algorithms (simple, Otsu, wavelet) on a randomly selected image.

  Params:
  -------
    files - List[str]: List of image file paths.

  Returns:
  --------
    None
  """
  f = np.random.choice(files)
  img = tifffile.imread(f)
  img = channel_normalize(img)
  simple_mask = simple_threshold(img)
  otsu_mask = otsu_thresholding(img)
  wavelet_mask = wavelet_thresholding(img)
  img_rgb = make_composite(img, luts=["magenta", "cyan"], ranges=[(0,1), (0,1)])
  simple_rgb = make_composite(simple_mask, luts=["magenta", "cyan"], ranges=[(0,1), (0,1)])
  otsu_rgb = make_composite(otsu_mask, luts=["magenta", "cyan"], ranges=[(0,1), (0,1)])
  wavelet_rgb = make_composite(wavelet_mask, luts=["magenta", "cyan"], ranges=[(0,1), (0,1)])
  fig, axs = plt.subplots(1, 4, figsize=(16,12))
  axs[0].imshow(img_rgb)
  axs[1].imshow(simple_rgb)
  axs[2].imshow(otsu_rgb)
  axs[3].imshow(wavelet_rgb)
  for title, ax in zip(["Image", "Simple", "Otsu", "Wavelet"], axs):
    ax.set_title(title)
    ax.axis("off")
  plt.show()


### You can uncomment selected lines below to test your implementations individually
# f = np.random.choice(files)
# img = tifffile.imread(f)
# img = channel_normalize(img)
# # _ = simple_threshold(img=img, display=True)
# # _ = otsu_thresholding(img=img, display=True)
# # _ = wavelet_thresholding(img=img, display=True)

### Uncomment code below to compare algorithms in a single figure
threshold_comparisons(files=files)

## 3.2 — Extract Protein Areas

**Context:** Now that we have segmented all proteins from an image, we can extract morphological features from the proteins using `skimage.measure.label` and `skimage.measure.regionprops` to identify individual protein clusters and measure their properties. We start with the **area** feature.

**Notes:**
- `skimage.measure.label` converts a simple [0,1] segmentation mask into a "labeled" mask where each protein has its unique identifier.
- `skimage.measure.regionprops` takes this labeled mask and extracts a Region Properties object from each, which contains most of the morphological features we care about.

**Steps:**
1. Implement `extract_area_from_prop` which returns `prop.area` from a regionprops object.
2. Implement `extract_areas_from_image`: label the mask, compute regionprops, then collect areas for all proteins in the image via `extract_area_from_prop`.
3. Implement `extract_protein_areas`: loop over all files, load and normalize each image, detect spots with `detect_spots`, and aggregate areas across images.

**Key functions:** `measure.label`, `measure.regionprops`, `detect_spots`, `channel_normalize`  
**Available variables:** `files`

In [ ]:
### Exercise 3.2 ###
from skimage import measure
from tqdm import tqdm

FEATURES = ["area"]

def extract_area_from_prop(prop, img: np.ndarray):
  """
  Extracts the area feature from a given region property.

  Params:
  -------
    prop: A region property object from skimage.measure.regionprops.
    img - np.ndarray: The intensity image (not directly used for area, but passed for consistency).

  Returns:
  --------
    float: The area of the region.
  """
  return # TODO

def extract_areas_from_image(img: np.ndarray, mask: np.ndarray) -> List[int]:
  """
  Extracts areas of all regions from a given image and its mask.

  Params:
  -------
    img - np.ndarray: The original image (intensity image).
    mask - np.ndarray: The binary mask indicating regions of interest.

  Returns:
  --------
    List[float]: A list of areas for each detected region.
  """
  label_image = measure.label(...) # TODO
  regionprops = measure.regionprops(...) # TODO
  image_areas = []
  for prop in regionprops:
    area = extract_area_from_prop(...) # TODO
    image_areas.append(area)
  return image_areas


def extract_protein_areas(files: List[str]) -> np.ndarray:
  """
  Extracts protein areas from a list of image files using wavelet detection and region properties.

  Params:
  -------
    files - List[str]: List of image file paths.

  Returns:
  --------
    np.ndarray: A numpy array containing all extracted protein areas.
  """
  protein_areas = []
  for f in tqdm(files, desc="... Extracting protein areas ..."):
    img = tifffile.imread(f)
    img = channel_normalize(img)
    mask = ... # TODO
    img_areas = extract_areas_from_image(...) # TODO
    protein_areas.extend(img_areas)
  return np.array(protein_areas)



def plot_histogram(data: np.ndarray) -> None:
  """
  Plots a histogram of the given data, setting x-axis limits based on the 99th percentile.

  Params:
  -------
    data - np.ndarray: The data to be plotted.

  Returns:
  --------
    None
  """
  fig = plt.figure(figsize=(4,4))
  ax = fig.add_subplot(111)
  ax.hist(data, bins=50, alpha=0.4, edgecolor="black")
  ax.set_xlim(0, np.quantile(data, 0.99))
  ax.set_xlabel("Area (pixels)")
  ax.set_ylabel("Counts")
  plt.show()

In [ ]:
## This line of code takes a little bit of time so we isolate it into its own cell.
protein_areas = extract_protein_areas(files=files)

In [ ]:
plot_histogram(protein_areas)

## 3.3 — Areas by Experimental Condition

**Context:** Now that we can extract areas from all images, we filter by experimental condition to enable comparisons between **Block** and **GluGly**. The provided `plot_histograms` and `plot_cumulative_frequencies` helpers will visualize the results.

**Steps:**
1. Implement `extract_protein_areas_by_condition`: filter `files` by condition using `get_condition_from_filename`, then apply the same pipeline as `extract_protein_areas`.
2. Call the function for both conditions and compare the area distributions using the provided `plot_histograms` and `plot_cumulative_frequencies` functions. Is one of the graphs better than the other? If so, why do you think that is?

**Key functions:** `get_condition_from_filename`, `extract_areas_from_image`, `detect_spots`  
**Available variables:** `files`, `extract_areas_from_image`

In [ ]:
def extract_protein_areas_by_condition(files: List[str], condition_filter: str = "Block") -> np.ndarray:
  """
  Extracts protein areas from image files that match a specified experimental condition.

  Params:
  -------
    files - List[str]: List of all image file paths.
    condition_filter - str: The experimental condition to filter files by (e.g., "Block", "GluGly").

  Returns:
  --------
    np.ndarray: A numpy array containing all extracted protein areas for the filtered condition.
  """
  protein_areas  = []
  filtered_files = [...] # TODO Filter files using condition_filter
  for f in tqdm(filtered_files, desc=f"... Extracting {condition_filter} protein areas ..."):
    img = tifffile.imread(f)
    img = channel_normalize(img)
    mask = ... # TODO
    img_features = extract_areas_from_image(...) # TODO
    protein_areas.extend(img_features)
  return np.array(protein_areas)


In [ ]:
block_areas = extract_protein_areas_by_condition(files=files, condition_filter="Block")
glugly_areas = extract_protein_areas_by_condition(files=files, condition_filter="GluGly")

In [ ]:
def plot_histograms(data_c1: np.ndarray, data_c2: np.ndarray) -> None:
  """
  Plots normalized histograms for two datasets, representing different conditions, with a common x-axis limit.

  Params:
  -------
    data_c1 - np.ndarray: Data for the first condition.
    data_c2 - np.ndarray: Data for the second condition.

  Returns:
  --------
    None
  """
  fig = plt.figure(figsize=(4, 4))
  ax = fig.add_subplot(111)

  counts_c1, bins_c1 = np.histogram(data_c1, bins=50)
  counts_c2, bins_c2 = np.histogram(data_c2, bins=50)

  # Normalize counts to have a maximum of 1.0
  normalized_counts_c1 = counts_c1 / counts_c1.max() if counts_c1.max() > 0 else counts_c1
  normalized_counts_c2 = counts_c2 / counts_c2.max() if counts_c2.max() > 0 else counts_c2
  ax.hist(bins_c1[:-1], bins_c1, weights=normalized_counts_c1, alpha=0.4, color="dodgerblue", edgecolor="black", label="Block")
  ax.set_xlim(0, np.quantile(data_c1, 0.99))
  ax.hist(bins_c2[:-1], bins_c2, weights=normalized_counts_c2, alpha=0.4, color="fuchsia", edgecolor="black", label="GluGly")

  ax.set_xlabel("Area")
  ax.set_ylabel("Normalized Counts")
  ax.legend()
  plt.title("Normalized Area Distribution by Condition")
  plt.show()

plot_histograms(data_c1=block_areas, data_c2=glugly_areas)

In [ ]:
def plot_cumulative_frequencies(data_c1: np.ndarray, data_c2: np.ndarray) -> None:
  """
  Plots the cumulative frequency distributions for two datasets.

  Params:
  -------
    data_c1 - np.ndarray: Data for the first condition.
    data_c2 - np.ndarray: Data for the second condition.

  Returns:
  --------
    None
  """
  fig = plt.figure(figsize=(4,4))
  ax = fig.add_subplot(111)

  sorted_data_c1 = np.sort(data_c1)
  sorted_data_c2 = np.sort(data_c2)

  cumulative_freq_c1 = np.arange(1, len(sorted_data_c1) + 1) / len(sorted_data_c1)
  cumulative_freq_c2 = np.arange(1, len(sorted_data_c2) + 1) / len(sorted_data_c2)

  ax.plot(sorted_data_c1, cumulative_freq_c1, color="dodgerblue", label="Block")
  ax.plot(sorted_data_c2, cumulative_freq_c2, color="fuchsia", label="GluGly")

  ax.set_xlabel("Area")
  ax.set_ylabel("Cumulative frequency")
  ax.set_xlim(0, np.quantile(data_c1, 0.99))
  ax.legend()
  plt.show()

plot_cumulative_frequencies(data_c1=block_areas, data_c2=glugly_areas)

# Exercise 4: Multidimensional Analysis

Area alone (or any single feature) may not allow use to distinguish the two conditions. In this exercise we will extract a richer **feature vector** per cluster, containing area, mean intensity, eccentricity, solidity, and local density. In this feature space, we use unsupervised clustering (K-Means) to identify protein subtypes. We then compare subtype frequencies between conditions. **Note that we will only be using the second channel of the images in this exercise to analyze the morphological features of PSD-95 specifically.**

## Exercise 4.1 — Multidimensional Feature Extraction

**Context:** `compute_density_and_nanodomains` (provided below) computes the local cluster density and number of fluorescence peaks (nanodomains) within a 1 µm² region around each protein. You will use it inside `extract_features_by_condition` to build a per-protein feature matrix. We provide these functions as they compute features which are not part of a RegionProperties object.

**Steps:**
1. In `extract_features_by_condition`: filter files by condition, load each image, detect spots, run `regionprops` on the last channel, and for each region collect `[area, mean_intensity, eccentricity, solidity, density]` using `compute_density_and_nanodomains`.
2. Stack per-image arrays with `np.concatenate` to build the final `(N_proteins, 5)` matrix.

The print statements 3 cells down show the shape of the extracted feature vectors, and should output the following:

`Block features: (12708, 5)`

`GluGly features: (12525, 5)`

`All features: (25233, 5)`

`All labels: (25233,)`,

Rougly 12000 proteins for each condition, and 5 features per protein.

**Key functions:** `measure.label`, `measure.regionprops`, `compute_density_and_nanodomains`, `np.concatenate`  
**Available variables:** `FEATURES`, `files`, `get_condition_from_filename`, `channel_normalize`, `detect_spots`

In [ ]:
from skimage.feature import peak_local_max

FEATURES = ["area", "Mintensity", "eccentricity", "solidity", "density"]

##### Exercise 4.1 #####
def compute_density_and_nanodomains(prop, mask: np.ndarray, img: np.ndarray):
  """
  Computes the density and number of nanodomains within a specified micron region around a protein.

  Params:
  -------
    prop: A region property object from skimage.measure.regionprops, containing the weighted centroid and slice.
    mask - np.ndarray: The binary mask of the protein.
    img - np.ndarray: The original intensity image.

  Returns:
  --------
    Tuple[float, int]: A tuple containing the density and the number of nanodomains.
  """
  y, x = [int(item) for item in prop.weighted_centroid]
  micron_region = mask[y - 25:y+25, x-25:x+25]
  _, num_in_region = measure.label(micron_region, return_num=True)
  density = num_in_region / (50**2 * 20**2) * (1000**2)
  min_distance = int(0.08 / 0.020) // 2 + 1
  slc_img = img[prop.slice]
  peaks = peak_local_max(slc_img, min_distance=min_distance, exclude_border=False, labels=prop.image)
  return density, len(peaks)


def extract_features_by_condition(files: List[str], condition_filter: str) -> np.ndarray:
  """
  Extracts a set of features (area, mean intensity, eccentricity, solidity, density) from proteins
  in images that match a specified experimental condition.

  Params:
  -------
    files - List[str]: List of all image file paths.
    condition_filter - str: The experimental condition to filter files by (e.g., "Block", "GluGly").

  Returns:
  --------
    np.ndarray: A numpy array containing all extracted protein features for the filtered condition.
  """
  all_features = None
  filtered_files = [f for f in files if get_condition_from_filename(f) == condition_filter]
  for file_idx, f in enumerate(tqdm(filtered_files, desc=f"... Extracting {condition_filter} protein features ...")):
    img = tifffile.imread(f)
    img = channel_normalize(img)
    mask = detect_spots(img)
    img, mask = img[-1], mask[-1]
    label_image, num_proteins = measure.label(mask, return_num=True)
    regionprops = measure.regionprops(label_image, intensity_image=img)
    image_features = np.zeros((num_proteins, len(FEATURES)))
    for i, prop in enumerate(regionprops):
      area = prop.area
      intensity = prop.mean_intensity
      eccentricity = prop.eccentricity
      solidity = prop.solidity
      try:
        density, nanodomains = compute_density_and_nanodomains(prop = prop, img=img, mask = mask )
      except ValueError:
        continue
      image_features[i] = [area, density, eccentricity, intensity, solidity]
    if file_idx == 0:
      all_features = image_features
    else:
      all_features = np.concatenate([all_features, image_features], axis=0)
  return all_features


In [ ]:
block_features = extract_features_by_condition(files=files, condition_filter="Block")
glugly_features = extract_features_by_condition(files=files, condition_filter="GluGly")

In [ ]:
print(block_features.shape, glugly_features.shape)
block_labels = ["Block"] * len(block_features)
glugly_labels = ["GluGly"] * len(glugly_features)
all_features = np.concatenate([block_features, glugly_features], axis=0)
all_labels = block_labels + glugly_labels
all_labels = np.array(all_labels)
print(all_features.shape)
print(all_labels.shape)


## Exercise 4.2 — Protein Subtype Discovery with K-Means

**Context:** We now cluster proteins in the feature space we computed in the previous exercise to discover synaptic subtypes which describe our data well. We do so using the K-Means algorithm, depicted below:


In [ ]:
image_path = f'{home}/images/kmeans.png'
display(Image(filename=image_path))

 You will now implement the `extract_subtypes` function which runs the K-Means algorithm. With K-Means, there is always the difficulty of choosing the value of **K**. To do this, we typically measure a clustering metric over a range of values of K, and select the value which optimizes this metric. In `extract_subtypes`, this can be done by setting `optimize_K=True` and `K=None`. Once this value of K is determined, we re-run the function with `optimize_K=False` and `K=optimal_K`. A metric commonly used is the silhouette score, you can check its documentation in the sklearn.metrics package if you are interested.

**Steps:**
1. Whether optimize_K is True or False, we must always normalize or standardize the input features. We do this because as K-Means relies heavily on distances in feature space, we don't want the result to be dominated by features which naturally have a large extent. You can standardize features to have a mean of 0 and a standard deviation of 1 with `StandardScaler`.
2. In `optimize_K` mode: iterate over K values, fit `KMeans`, compute `silhouette_score`, plot the curve, and return `argmax + 3`, as the first value of K considered is 3.
3. In fixed-K mode: fit `KMeans(n_clusters=K)` and return the fitted clusterer.

There are helper functions to display the average features of proteins belonging to each computed subtypes in later cells.

**Key functions:** `StandardScaler`, `KMeans`, `silhouette_score`  
**Available variables:** `all_features`, `all_labels` (built in the execution cell below)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

##### Exercise 4.2 #####
def extract_subtypes(features: np.ndarray, labels: np.ndarray, K: int = None, optimize_K: bool = True) -> np.ndarray | int:
  """
  Extracts protein subtypes using K-Means clustering. Optionally optimizes for the best K based on silhouette scores.

  Params:
  -------
    features - np.ndarray: A 2D numpy array where each row is a feature vector for a protein.
    labels - np.ndarray: A 1D numpy array of labels (conditions) for the proteins.
    K - int: The number of clusters to use if `optimize_K` is False.
    optimize_K - bool: If True, the function will determine the best K (number of clusters)
                        based on the silhouette score and return it. Otherwise, it performs clustering
                        with the given K and returns the KMeans clusterer.

  Returns:
  --------
    np.ndarray | int: If `optimize_K` is True, returns the optimal number of clusters (int).
                      If `optimize_K` is False, returns the fitted KMeans clusterer object.
  """
  scaler = StandardScaler()
  scaled_features = ... # TODO: standardize features using scaler.fit_transform(...)
  if optimize_K:
    num_clusters = list(range(3, 10))
    sil_scores = []
    for nc in tqdm(num_clusters, desc="... Finding best K for K-Means..."):
      clusterer = KMeans(...) # TODO
      cluster_labels = clusterer.fit_predict(...) # TODO
      score = silhouette_score(...)
      sil_scores.append(score)
    x = [item + 3 for item in np.arange(len(num_clusters))]
    fig = plt.figure(figsize=(4,4))
    ax = fig.add_subplot(111)
    ax.plot(x, sil_scores, marker='.')
    ax.set_ylabel("Silhouette scores")
    ax.set_xlabel("K")
    plt.show()
    return np.argmax(np.array(sil_scores)) + 3
  else:
    clusterer = KMeans(...)
    clusterer = clusterer.fit(...)
    return clusterer

best_K = extract_subtypes(features=all_features, labels=all_labels)


In [ ]:
### Here we extract the centroids and the labels from the clusterer to display the average feature vectors for each condition in the next cell

clusterer = extract_subtypes(features=all_features, labels=all_labels, K=best_K, optimize_K=False)
subtypes = clusterer.cluster_centers_
cluster_labels = clusterer.labels_

In [ ]:
def plot_feature_vectors(subtypes: np.ndarray, cmap=plt.cm.RdPu):
  """
  Plots a heatmap of feature vectors representing different protein subtypes.

  Params:
  -------
    subtypes - np.ndarray: A 2D numpy array where each row is a feature vector for a subtype.
    cmap - matplotlib.colors.Colormap: Colormap to use for the heatmap.

  Returns:
  --------
    None
  """
  fig = plt.figure(figsize=(4,4))
  ax = fig.add_subplot(111)
  ax.imshow(subtypes, cmap=cmap)
  ax.set_yticks(np.arange(len(subtypes)))
  ax.set_xticks(np.arange(subtypes.shape[1]))
  ax.set_xticklabels(FEATURES, rotation=45)
  ax.set_ylabel("Subtypes")
  plt.show()

plot_feature_vectors(subtypes=subtypes)

## 4.3 — Subtype Frequencies per Condition

**Context:** After clustering, we now check whether certain protein subtypes are enriched in one condition. For each condition, we count how many proteins belong to each cluster and normalize by the total number of proteins in that condition.

**Steps:**
1. Build a `counts` dict (condition → cluster_id → count) by iterating over `zip(labels, cluster_labels)`.
2. Normalize counts for each condition by their total (`np.sum`).
3. Display as a grouped bar chart, one bar group per subtype, one color per condition.

**Key functions:** `np.unique`, `ax.bar`  
**Available variables:** `all_labels`, `clusterer.labels_` (from Exercise 4.2)

In [ ]:
##### Exercise 4.3 ######

def subtype_frequencies(labels: np.ndarray, cluster_labels: np.ndarray) -> None:
  """
  Calculates and plots the normalized frequencies of protein subtypes for each experimental condition.

  Params:
  -------
    labels - np.ndarray: A 1D numpy array of original condition labels for each protein.
    cluster_labels - np.ndarray: A 1D numpy array of cluster assignments for each protein.

  Returns:
  --------
    None
  """
  num_subtypes = len(np.unique(cluster_labels))
  counts = {
      condition: {key: 0 for key in np.arange(num_subtypes)}
      for condition in ["Block", "GluGly"]
  }

  for l, cl in zip(labels, cluster_labels):
    counts[...][...] += 1 # TODO: Add one count to the correct dictionary entry
  block_counts  = np.array(list(counts["Block"].values()))
  glugly_counts = np.array(list(counts["GluGly"].values()))
  block_counts  = block_counts  / ... # TODO: Normalize by the total number of proteins per condition, so that the counts sum to 1.0
  glugly_counts = glugly_counts / ... # TODO: Normalize by the total number of proteins per condition, so that the counts sum to 1.0
  fig = plt.figure(figsize=(4,4))
  ax = fig.add_subplot(111)
  x1 = np.arange(num_subtypes)
  width = 0.25
  x2 = [item + width for item in x1]
  ax.bar(x1, block_counts,  width=width, color=..., label=...) # TODO: fill in the arguments to properly display the bar chart
  ax.bar(x2, glugly_counts, width=width, color=..., label=...) # TODO: fill in the arguments to properly display the bar chart
  ax.set_xlabel("Subtypes")
  ax.set_ylabel("Proportion")
  ax.legend()
  plt.show()

subtype_frequencies(labels=all_labels, cluster_labels=cluster_labels)